In [1]:
import pandas as pd
import os
from datetime import datetime, timezone

print("Gold pipeline started...")

Gold pipeline started...


In [2]:
# LOAD SILVER DATA


silver_path = "silver/flights_clean.parquet"

if not os.path.exists(silver_path):
    raise FileNotFoundError(f"Silver dataset not found: {silver_path}")

df = pd.read_parquet(silver_path)

if df.empty:
    raise ValueError("Silver dataset is empty")

print(f"Silver data loaded | Rows: {df.shape[0]} | Columns: {df.shape[1]}")

Silver data loaded | Rows: 463484 | Columns: 37


In [3]:
# CREATE GOLD DIRECTORY


gold_root = "gold"
os.makedirs(gold_root, exist_ok=True)

In [4]:
# AGG 1 — AVERAGE DELAY BY AIRLINE


print("\nComputing airline delay metrics...")

agg1 = (
    df.groupby("airline")
    .agg(
        avg_arr_delay=("arr_delay", "mean"),
        avg_dep_delay=("dep_delay", "mean"),
        total_flights=("fl_number", "count")
    )
    .reset_index()
)

agg1["avg_arr_delay"] = agg1["avg_arr_delay"].round(2)
agg1["avg_dep_delay"] = agg1["avg_dep_delay"].round(2)

agg1 = agg1.sort_values("avg_arr_delay", ascending=False)

print(f"Agg1 complete | Rows: {agg1.shape[0]}")


Computing airline delay metrics...
Agg1 complete | Rows: 15


In [5]:
# AGG 2 — CANCELLATION RATE BY AIRLINE


print("\nComputing cancellation metrics...")

agg2 = (
    df.groupby("airline")
    .agg(
        total_flights=("fl_number", "count"),
        total_cancelled=("cancelled", "sum")
    )
    .reset_index()
)

agg2["cancellation_rate_pct"] = (
    (agg2["total_cancelled"] / agg2["total_flights"]) * 100
).round(2)

agg2 = agg2.sort_values("cancellation_rate_pct", ascending=False)

print(f"Agg2 complete | Rows: {agg2.shape[0]}")


Computing cancellation metrics...
Agg2 complete | Rows: 15


In [6]:
# AGG 3 — DELAY CAUSE BREAKDOWN


print("\nComputing delay cause breakdown...")

delay_cols = [
    "delay_due_carrier",
    "delay_due_weather",
    "delay_due_nas",
    "delay_due_security",
    "delay_due_late_aircraft"
]

agg3 = df[delay_cols].sum().reset_index()

agg3.columns = ["delay_cause", "total_delay_minutes"]

agg3["total_delay_minutes"] = agg3["total_delay_minutes"].round(0)

agg3 = agg3.sort_values("total_delay_minutes", ascending=False)

print(f"Agg3 complete | Rows: {agg3.shape[0]}")


Computing delay cause breakdown...
Agg3 complete | Rows: 5


In [7]:
# AGG 4 — MOST DELAYED ROUTES


print("\nComputing most delayed routes...")

agg4 = (
    df.groupby(["origin", "dest"])
    .agg(
        avg_arr_delay=("arr_delay", "mean"),
        total_flights=("fl_number", "count")
    )
    .reset_index()
)

agg4["avg_arr_delay"] = agg4["avg_arr_delay"].round(2)

agg4 = agg4[agg4["total_flights"] >= 10]

agg4 = agg4.sort_values("avg_arr_delay", ascending=False).head(20)

print(f"Agg4 complete | Rows: {agg4.shape[0]}")


Computing most delayed routes...
Agg4 complete | Rows: 20


In [8]:
# AGG 5 — MONTHLY DELAY TREND


print("\nComputing monthly delay trend...")

agg5 = (
    df.groupby("month")
    .agg(
        avg_arr_delay=("arr_delay", "mean"),
        avg_dep_delay=("dep_delay", "mean"),
        total_flights=("fl_number", "count")
    )
    .reset_index()
)

agg5["avg_arr_delay"] = agg5["avg_arr_delay"].round(2)
agg5["avg_dep_delay"] = agg5["avg_dep_delay"].round(2)

agg5 = agg5.sort_values("month")

print(f"Agg5 complete | Rows: {agg5.shape[0]}")


Computing monthly delay trend...
Agg5 complete | Rows: 8


In [9]:
# PREPARE DATASETS


datasets = {
    "avg_delay_by_airline": agg1,
    "cancellation_by_airline": agg2,
    "delay_cause_breakdown": agg3,
    "most_delayed_routes": agg4,
    "monthly_delay_trend": agg5
}

In [10]:
# ADD PIPELINE TIMESTAMP


run_timestamp = datetime.now(timezone.utc)

for name, data in datasets.items():
    data["ingestion_timestamp"] = run_timestamp

In [11]:
# SAVE GOLD DATASETS


print("\nSaving gold datasets...")

for name, data in datasets.items():

    folder = f"{gold_root}/{name}"
    os.makedirs(folder, exist_ok=True)

    path = f"{folder}/data.parquet"

    data.to_parquet(path, index=False)

    print(f"Saved: {path} | Rows: {data.shape[0]}")

print("\n Gold pipeline completed successfully!")


Saving gold datasets...
Saved: gold/avg_delay_by_airline/data.parquet | Rows: 15
Saved: gold/cancellation_by_airline/data.parquet | Rows: 15
Saved: gold/delay_cause_breakdown/data.parquet | Rows: 5
Saved: gold/most_delayed_routes/data.parquet | Rows: 20
Saved: gold/monthly_delay_trend/data.parquet | Rows: 8

 Gold pipeline completed successfully!
